# Lab 17: Classification — Diagnosing a Probability That Is Not One
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 115 min Core + 20 min Extension

---

**Learning Objectives:**
- Show that two models with nearly the same AUC can give probabilities of different quality, and explain why AUC cannot see the difference
- Read a reliability diagram, and measure calibration with the Brier score and with an expected calibration error you write yourself
- Show what training on a balanced sample does to a forest's probabilities, and repair them with Platt scaling and isotonic regression fitted on rows the forest never saw
- Show how the cost-minimising threshold moves when the probabilities are off, and price the miscalibration per 1,000 decisions
- Ship a module that reports discrimination and calibration separately

**Dataset:** 8,000 simulated loans with a known default probability (13.3% default)

**Format:** Each diagnosis shows you a result that is wrong, or wrong-looking, and asks what happened. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3), the blanks you fill in each diagnosis, the required `calibration_error()` function, and `src/classify_tools.py`, a module you finish with your `calibration_error()`, save and upload.

**Time:** Core (115 min, including the required Write it yourself) + Extension (optional, 20 min)

With setup, the README and the GitHub submission, plan on about two and a quarter hours, most of it after class. The optional Extension adds about 20 minutes.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Diagnosis 3:** two blanks that choose a threshold for the raw scores on the calibration rows.
> - **Write it yourself (required, after Diagnosis 3):** `calibration_error()`, the expected calibration error, about eight lines in an empty cell. A check cell tells you when it is right. In Ship It you paste it, unchanged, into the module.
> - **Extension (optional):** a rise in the default rate and an intercept correction, from a three-step outline.
>
> **You write, in text cells:** three answers in Diagnosis 1, three in Diagnosis 2 and two in Diagnosis 3.
>
> **You run and read:** everything else. That is the simulated loans and the two models, the reliability diagram, the count of predictions per tenth and the squared probabilities, the analyst's balanced forest, the split of the test rows, the Platt repair and the table of both repairs on both forests, the cost of a threshold rule, and the rest of Ship It's module, `classify_tools.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself turns a reliability diagram with equal-width bins into one number.

## Setup

The first line of the setup cell upgrades scikit-learn to 1.8 or later, because the expected
numbers in this lab were produced on 1.8 or later. The cell prints your scikit-learn version. If it
stops with an `ImportError`, or shows a version below 1.8, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries and the seed
!pip install "scikit-learn>=1.8" -q

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn import metrics
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Path("src").mkdir(exist_ok=True)     # Ship It writes its module into this folder

RANDOM_STATE = 42                    # the seed for every split and every forest
print(f"scikit-learn {sklearn.__version__} (this lab needs 1.8 or later)")
print("Setup complete.")

## Diagnosis 1: Ranks Well, Lies About Probability (30 min)

A loan model is presented as ready for a decision rule that uses its probabilities: decline a
loan when its predicted chance of default is high enough. The evidence offered is an AUC above
0.8. Ranking applicants well and giving probabilities you can use as numbers are different claims,
and this diagnosis separates them. You know the true default probabilities, because you are about
to simulate them: 8,000 loans, six features, and a logistic model in which four features matter.

**Reading the code.** `rng.normal(0, 1, (n, 6))` draws a table of n rows and 6 columns of standard
normal numbers. `@` is matrix multiplication, so `X @ np.array([...])` adds up the six columns,
each times its number; the two zeros mean columns `X[:, 3]` and `X[:, 4]` do nothing. `1 / (1 + np.exp(-lin))`
turns that index into a probability (the logistic function), and `rng.binomial(1, prob)` draws a
1, a default, with that probability for each row. The mean of 0s and 1s is the share of 1s, here
the *prevalence*, the share of loans that default; `{x:.1%}` prints a share as a percentage with
one decimal.

In [ ]:
# GIVEN — run as-is
# 8,000 simulated loans: six features, and defaults drawn from a logistic model
rng = np.random.default_rng(RANDOM_STATE)
n = 8_000
X = rng.normal(0, 1, (n, 6))
lin = -2.6 + X @ np.array([1.1, -0.8, 0.6, 0.0, 0.0, 0.3])
y = rng.binomial(1, 1 / (1 + np.exp(-lin)))
print(f"prevalence = {y.mean():.1%}")

**Expected output:** `prevalence = 13.3%`.

**Reading the code.** `train_test_split(..., test_size=0.3, stratify=y)` holds out 30% of the
rows, 2,400 loans, as a test set, with the same share of defaults in both parts. `make_pipeline(
StandardScaler(), LogisticRegression(...))` standardises the features, then fits a logistic
regression; `max_iter` lets the solver take more steps. The forest grows 200 trees, each at most 6
splits deep (`max_depth=6`). Both models are fitted on the training rows only.

In [ ]:
# GIVEN — run as-is
# Hold out 30% of the loans, then fit a logistic regression and a random forest on the rest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=RANDOM_STATE, stratify=y)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=200, max_depth=6,
                                random_state=RANDOM_STATE).fit(X_train, y_train)

**Reading the code.** `.predict_proba(X_test)` gives two columns per row, the probability of 0 and
the probability of 1; `[:, 1]` keeps the second, the predicted probability of default.
`metrics.roc_auc_score(y_true, p)` is the AUC. `metrics.brier_score_loss(y_true, p)` is the Brier
score, the mean squared gap between each outcome (0 or 1) and its predicted probability, so lower
is better. `{name:>15}` right-aligns the name in 15 characters, and `{x:.4f}` prints 4 decimals.

In [ ]:
# GIVEN — run as-is
# Each model's predicted default probabilities on the test rows, and two scores of them
p_logit = logit.predict_proba(X_test)[:, 1]
p_forest = forest.predict_proba(X_test)[:, 1]
for name, p in [("logistic", p_logit), ("random forest", p_forest)]:
    print(f"{name:>15}: AUC {metrics.roc_auc_score(y_test, p):.4f}   "
          f"Brier {metrics.brier_score_loss(y_test, p):.4f}")

**Expected output:** the logistic model has an AUC of 0.8389 and a Brier score of 0.0889; the
random forest has an AUC of 0.8232 and a Brier score of 0.0927.

**The reliability diagram** asks whether a predicted 20% happens 20% of the time.
`calibration_curve(y_true, p, n_bins=10, strategy="quantile")` sorts the test rows by predicted
probability, cuts them into 10 groups of equal size, and returns two arrays: for each group, the
share that defaulted and the mean predicted probability. A point on the dashed diagonal is a group
whose predictions match its default rate. A point below it is a group where the model promised more
risk than it delivered; a point above it, less.

In [ ]:
# GIVEN — run as-is
# Reliability diagram: observed default rate against mean predicted probability, 10 groups per model
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
for name, p in [("logistic", p_logit), ("random forest", p_forest)]:
    observed, predicted = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, "o-", label=name)
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed default rate")
ax.legend()
plt.show()

**Expected output:** a dashed diagonal and two curves of 10 points each. The forest's last point
is at a mean predicted probability of about 0.43.

**Where the predictions fall.** The next cell counts each model's test predictions in ten bins of
equal width: 0 to 0.1, 0.1 to 0.2, and so on up to 0.9 to 1. `np.floor(p * 10)` rounds each
prediction times 10 down to a whole number, which is its bin number from 0 to 9, and
`.astype(int)` stores it as an integer. A probability of exactly 1.0 would get bin 10, so
`np.minimum(..., 9)` puts it in the top bin. `np.bincount(bins, minlength=10)` counts how many
predictions carry each bin number from 0 to 9, and `p.max()` is the largest prediction.

In [ ]:
# GIVEN — run as-is
# How many test predictions fall in each tenth of [0, 1], and the largest prediction
for name, p in [("logistic", p_logit), ("random forest", p_forest)]:
    bins = np.minimum(np.floor(p * 10).astype(int), 9)      # a probability of 1.0 goes in the top bin
    print(f"{name:>15}: {np.bincount(bins, minlength=10)}   largest {p.max():.3f}")

**Expected output:**
```
       logistic: [1494  406  218   98   56   54   33   23   13    5]   largest 0.947
  random forest: [1385  589  218   91   62   33   16    6    0    0]   largest 0.735
```

**A change that keeps the order.** Halving the predictions, or squaring them, keeps their order
and gives a new set of scores between 0 and 1. The next cell squares the forest's test
probabilities and scores them as the cell before the diagram scored the raw ones.

In [ ]:
# GIVEN — run as-is
# The forest's probabilities squared: the same order, smaller numbers
squared = p_forest ** 2
print(f"squared forest: AUC {metrics.roc_auc_score(y_test, squared):.4f}   "
      f"Brier {metrics.brier_score_loss(y_test, squared):.4f}")

**Expected output:** `squared forest: AUC 0.8232   Brier 0.1111`.

### YOUR DIAGNOSIS

1. Squaring the forest's probabilities left its AUC exactly where it was and moved its Brier score from 0.0927 to 0.1111. Explain what AUC measures, why no change that keeps the order of the predictions can move it, and what the Brier score looks at that AUC does not.
2. On the same test rows, the forest's predictions stop at 0.735 and the logistic model's reach 0.947. Explain why averaging 200 trees, each at most 6 splits deep, pulls the extreme probabilities toward the middle, and where that shows on the reliability diagram.
3. The bank's rule declines a loan when its predicted probability is above a cut-off computed from costs. Which does that rule need: a good ranking, or probabilities that mean what they say? Justify it from the diagram.

*Your answers here:*

---

## Diagnosis 2: Repair, and What It Costs (25 min)

Only 13.3% of the loans default, and the data team worries that the forest sees too few defaults
to learn from. Their fix is a common one: grow the forest again on a *balanced* sample, every
default in the training rows and the same number of repaid loans drawn at random, so that half of
the sample defaults. They report the new AUC and stop there.

**Reading the code.** `np.flatnonzero(y_train == 1)` gives the positions of the training rows
that defaulted. `sampler.choice(a, size=k, replace=False)` draws k different entries of `a` at
random, from a generator with a fixed seed. `np.concatenate([a, b])` joins two arrays end to end,
and `X_train[balanced]` keeps the rows at those positions. Apart from its rows, this is Diagnosis
1's forest. `{x:.0%}` prints a percentage with no decimals.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Diagnosis 1's forest again, grown on a balanced sample: every default, and as many repaid loans
sampler = np.random.default_rng(RANDOM_STATE)
defaults = np.flatnonzero(y_train == 1)
repaid = sampler.choice(np.flatnonzero(y_train == 0), size=len(defaults), replace=False)
balanced = np.concatenate([defaults, repaid])
forest_bal = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE).fit(
    X_train[balanced], y_train[balanced])
print(f"balanced sample: {len(balanced)} loans, {y_train[balanced].mean():.0%} of them defaults")

**Expected output:** `balanced sample: 1486 loans, 50% of them defaults`.

The next cell scores both forests on the test rows. `p.mean()` is the mean predicted
probability. For calibrated probabilities, it is close to the share of loans that
default.

In [ ]:
# GIVEN — run as-is
# Diagnosis 1's forest and the balanced forest on the same test rows
p_bal = forest_bal.predict_proba(X_test)[:, 1]
print(f"test prevalence: {y_test.mean():.1%}")
for name, p in [("random forest", p_forest), ("balanced forest", p_bal)]:
    print(f"{name:>15}: AUC {metrics.roc_auc_score(y_test, p):.4f}   "
          f"Brier {metrics.brier_score_loss(y_test, p):.4f}   mean predicted {p.mean():.1%}")

**Expected output:** a test prevalence of 13.3%. Diagnosis 1's forest: AUC 0.8232, Brier 0.0927,
mean predicted 12.7%. The balanced forest: AUC 0.8249, Brier 0.1657, mean predicted 38.8%.

### YOUR DIAGNOSIS

1. The balanced forest has about the same AUC and a far worse Brier score. Explain what training on a balanced sample did to its probabilities, and why AUC did not register it, using your first answer in Diagnosis 1.

*Your answers here:*

### Diagnosis 2, continued: the repair

The repair from class fits a second model that maps the forest's scores to observed default rates.
**Platt scaling** fits a logistic curve to the scores; **isotonic regression** fits a step function
that can only go up. Both are monotone: a higher score never gets a lower repaired probability.

The repair has to be fitted on rows that played no part in fitting the forest, and then scored on
rows that played no part in either fit. So the next cell splits the 2,400 test rows in half: 1,200
*calibration rows* to fit the repair, and 1,200 *evaluation rows* to score it. `y_cal.sum()`
counts the defaults, because the sum of 0s and 1s is the number of 1s.

In [ ]:
# GIVEN — run as-is
# Half of the test rows fit the repair (calibration rows); the other half score it (evaluation rows)
X_cal, X_eval, y_cal, y_eval = train_test_split(X_test, y_test, test_size=0.5,
                                                random_state=RANDOM_STATE, stratify=y_test)
print(f"calibration rows: {len(y_cal)}, of which {y_cal.sum()} defaults")
print(f"evaluation rows:  {len(y_eval)}, of which {y_eval.sum()} defaults")

**Expected output:** 1,200 calibration rows with 160 defaults, and 1,200 evaluation rows with 159.

**Reading the code.** `FrozenEstimator(forest_bal)` wraps the fitted forest so that nothing can
refit it. `CalibratedClassifierCV(..., method="sigmoid")` is Platt scaling: scikit-learn calls it
sigmoid, after the logistic curve. Its `.fit(X_cal, y_cal)` fits only the curve, on the calibration
rows, and its `.predict_proba` runs the forest and then the curve.

In [ ]:
# GIVEN — run as-is
# Platt scaling for the balanced forest: fitted on the calibration rows, scored on the evaluation rows
platt_bal = CalibratedClassifierCV(FrozenEstimator(forest_bal), method="sigmoid").fit(X_cal, y_cal)
p_bal_eval = forest_bal.predict_proba(X_eval)[:, 1]
p_platt = platt_bal.predict_proba(X_eval)[:, 1]
for name, p in [("raw", p_bal_eval), ("Platt", p_platt)]:
    print(f"{name:>6}: AUC {metrics.roc_auc_score(y_eval, p):.4f}   "
          f"Brier {metrics.brier_score_loss(y_eval, p):.4f}   mean predicted {p.mean():.1%}")

**Expected output:** on the evaluation rows, the balanced forest's raw probabilities have an AUC
of 0.8178, a Brier score of 0.1710 and a mean of 39.4%; after Platt scaling, 0.8178, 0.0930 and
14.3%.

**Is the repair worth it for every model?** The next cell repairs both forests, Diagnosis 1's and
the balanced one, with both methods, `"sigmoid"` (Platt) and `"isotonic"`, each fitted on the
calibration rows as the cell above did and scored on the evaluation rows. It prints each forest's
raw line first. `{'':>15}` prints 15 spaces, so each method's line sits under its forest's name.

In [ ]:
# GIVEN — run as-is
# Both repairs on both forests, each fitted on the calibration rows and scored on the evaluation rows
for name, model in [("random forest", forest), ("balanced forest", forest_bal)]:
    p_raw = model.predict_proba(X_eval)[:, 1]
    print(f"{name:>15} {'raw':>9}: AUC {metrics.roc_auc_score(y_eval, p_raw):.4f}   "
          f"Brier {metrics.brier_score_loss(y_eval, p_raw):.4f}")
    for method in ["sigmoid", "isotonic"]:
        repaired = CalibratedClassifierCV(FrozenEstimator(model), method=method).fit(X_cal, y_cal)
        p_rep = repaired.predict_proba(X_eval)[:, 1]
        print(f"{'':>15} {method:>9}: AUC {metrics.roc_auc_score(y_eval, p_rep):.4f}   "
              f"Brier {metrics.brier_score_loss(y_eval, p_rep):.4f}")

**Expected output:**
```
  random forest       raw: AUC 0.8201   Brier 0.0926
                  sigmoid: AUC 0.8201   Brier 0.0939
                 isotonic: AUC 0.8181   Brier 0.0936
balanced forest       raw: AUC 0.8178   Brier 0.1710
                  sigmoid: AUC 0.8178   Brier 0.0930
                 isotonic: AUC 0.8167   Brier 0.0939
```
### YOUR DIAGNOSIS

2. The repair cuts the balanced forest's Brier score almost in half, and makes Diagnosis 1's forest's Brier score slightly worse. Explain the difference: what does the repair spend, and when is it worth spending?
3. Platt's AUC equals the raw AUC to four decimals for both forests; isotonic's does not. Explain how a step function can move AUC a little, and what isotonic risks when the calibration rows hold 160 defaults.

*Your answers here:*

---

## Diagnosis 3: The Threshold Moves (20 min)

The bank flags a loan for decline when its predicted probability of default is at least a
threshold t. A default that is not flagged (a false negative) costs 8 times as much as a good loan
that is flagged (a false positive). From class, the threshold that minimises expected cost is
t\* = C_FP / (C_FP + C_FN) = 1 / (1 + 8) = 1/9 ≈ 0.111, and the derivation assumes the
probabilities are calibrated.

**Reading the code.** `cost_per_1000` flags every row whose probability is at least `threshold`.
`~flag` is "not flagged" and `&` is "and", applied row by row to arrays of True/False, and
`np.sum` counts the Trues. The cost is in units of one false positive, per 1,000 decisions.

In [ ]:
# GIVEN — run as-is
# The cost of a threshold rule, per 1,000 decisions, in units of one false positive
FN_COST, FP_COST = 8, 1
RULE_T = FP_COST / (FP_COST + FN_COST)          # 1/9, the threshold from class


def cost_per_1000(y_true, y_prob, threshold, fn_cost=FN_COST, fp_cost=FP_COST):
    """Cost per 1,000 decisions of flagging every row with a probability of at least threshold."""
    flag = y_prob >= threshold
    missed = np.sum(~flag & (y_true == 1))           # false negatives: defaults not flagged
    false_alarms = np.sum(flag & (y_true == 0))      # false positives: good loans flagged
    return 1000 * (fn_cost * missed + fp_cost * false_alarms) / len(y_true)

In [ ]:
# GIVEN — run as-is
# The rule t = 1/9 on the evaluation rows: the balanced forest's raw probabilities, then Platt's
for name, p in [("raw", p_bal_eval), ("Platt", p_platt)]:
    print(f"{name:>6} at t = {RULE_T:.3f}: flags {np.sum(p >= RULE_T)} of {len(p)}, "
          f"cost {cost_per_1000(y_eval, p, RULE_T):.1f} per 1,000")
    # average precision: the area under the precision-recall curve, a second ranking score
    print(f"{'':>6} AUC {metrics.roc_auc_score(y_eval, p):.4f}   "
          f"average precision {metrics.average_precision_score(y_eval, p):.4f}")

**Expected output:** at t = 0.111 the raw probabilities flag 1109 of the 1,200 loans at a cost of
791.7 per 1,000 decisions; Platt's flag 463 at a cost of 508.3. Both have an AUC of 0.8178 and an
average precision of 0.4320. Average precision, like AUC, depends only on the order of the
scores; its no-skill value is the prevalence rather than 0.5.

**Your task.** Without the repair, you could instead search for the threshold that suits the raw
scores. Fill in the two blanks: the cost of each threshold on the calibration rows, and the
position of the smallest cost (Lab 15 used `np.argmin` the same way). The cell then reports the
chosen threshold's cost on the evaluation rows, which played no part in choosing it.

**Reading the code.** `np.arange(0.01, 1.00, 0.01)` is 0.01, 0.02, ..., 0.99, and
`np.round(..., 2)` removes the tiny errors that stepping by 0.01 leaves in the last digits.
`[f(t) for t in THRESHOLDS]` builds a list with one entry per threshold.

In [ ]:
# YOUR TASK — Diagnosis 3: fill in the two blanks (____)
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)      # 0.01, 0.02, ..., 0.99
p_bal_cal = forest_bal.predict_proba(X_cal)[:, 1]
cal_costs = [____ for t in THRESHOLDS]                      # one cost per threshold, on the calibration rows
best_t = THRESHOLDS[____]
print(f"threshold chosen on the calibration rows: {best_t:.2f}")
print(f"   raw at t = {best_t:.2f}: flags {np.sum(p_bal_eval >= best_t)} of {len(p_bal_eval)}, "
      f"cost {cost_per_1000(y_eval, p_bal_eval, best_t):.1f} per 1,000")

**Expected output:** a threshold of 0.52, which on the evaluation rows flags 353 of 1,200 loans at
a cost of 491.7 per 1,000. A threshold of 0.49 at a cost of 480.0 means you searched the
evaluation rows: choose the threshold on the calibration rows. A threshold of 0.05 or 0.01 means
the outcomes and the probabilities come from different rows (`y_cal` with `p_bal_eval`, or `y_eval`
with `p_bal_cal`): both must be the calibration rows'. Python runs either without an error.

### YOUR DIAGNOSIS

1. Price the miscalibration: how much more per 1,000 decisions does the bank pay if it applies t = 1/9 to the balanced forest's raw probabilities rather than to Platt's? Explain why the rule fails on the raw probabilities.
2. The threshold you chose for the raw scores, 0.52, is far from 1/9, yet on the evaluation rows it costs 491.7 per 1,000, a little less than Platt's 508.3 at 1/9. Explain how a threshold so far from 1/9 can suit these scores. If a threshold search fixes the decision, why repair the probabilities at all? Give two uses of a predicted probability that a moved threshold cannot fix.

*Your answers here:*

---

## Write it yourself (required — 30 min)

This function is required. No AI: write it yourself.

A reliability diagram is a picture, and a model report needs a number. In the empty cell below,
write a function `calibration_error(y_true, y_prob, n_bins=10)` that returns the **expected
calibration error** of the probabilities `y_prob` for the outcomes `y_true`, two numpy arrays of the
same length:

1. Give each prediction a bin number: `n_bins` bins of equal width on [0, 1], with a probability
   of exactly 1.0 in the top bin. Diagnosis 1 did this for 10 bins when it counted the predictions
   per tenth; here the number of bins is the argument `n_bins`.
2. For each bin that holds at least one prediction, take the gap between the share of its rows that
   defaulted and its mean predicted probability, without the sign. A bin with no predictions is
   left out.
3. Return the average of those gaps, each weighted by the share of all the rows that falls in its
   bin.

Each gap is how far one point of a reliability diagram with equal-width bins sits from the
diagonal, so the function turns that diagram into one number, with each point counted by the
number of loans it stands for. It returns one number between 0 and 1.

**Tools.** A comparison such as `bins == b` gives an array of True/False, and `y_true[mask]` keeps
the rows where `mask` is True. The mean of 0s and 1s is a share, and so is the mean of True/False
(True counts as 1), so `mask.mean()` is the share of all rows that `mask` keeps. `np.sum(mask)`
counts them, and `abs(x)` drops the sign of x. You need about eight lines. Do not use
`calibration_curve`.

The check runs your function on a small example, where `y_true = [0, 0, 1, 1, 0, 1, 0]` and
`y_prob = [0.05, 0.15, 0.15, 0.35, 0.45, 0.85, 1.0]` should give 0.4286; then on the balanced
forest's evaluation probabilities from Diagnosis 2, raw and after Platt scaling; and with
`n_bins=5` and `n_bins=20`. Until you write the function, the check says "Not written yet"; once you do, it fails
with a message until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: write calibration_error(y_true, y_prob, n_bins=10)

In [ ]:
# CHECK — for calibration_error above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "calibration_error" not in globals():
    print("Not written yet: write calibration_error in the empty cell above, then run this check again.")
else:
    import inspect

    def names_used(code):
        """Every global name a function's code refers to, including inside functions it defines."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def near(a, b):
        """Equal to about six significant digits."""
        return bool(np.isclose(a, b, rtol=1e-6, atol=1e-12))

    def shown(value):
        """A short description of what calibration_error gave back."""
        if isinstance(value, dict):
            return "a dictionary with " + ", ".join(repr(k) for k in value)
        return f"a {type(value).__name__}"

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(calibration_error)
    params = list(signature.parameters)
    assert params[:3] == ["y_true", "y_prob", "n_bins"], (
        f"calibration_error's arguments are ({', '.join(params)}). The check calls it as the task "
        "asks: calibration_error(y_true, y_prob, n_bins=10).")
    assert signature.parameters["n_bins"].default == 10, (
        "Give n_bins a default of 10: def calibration_error(y_true, y_prob, n_bins=10).")
    used = names_used(calibration_error.__code__)
    assert "calibration_curve" not in used, (
        "calibration_error calls calibration_curve. Write the binning yourself.")
    for lab_name in ["y", "y_test", "y_eval", "y_cal", "p", "p_forest", "p_logit", "p_bal",
                     "p_bal_eval", "p_platt", "p_raw", "p_rep", "bins"]:
        assert lab_name not in used, (
            f"calibration_error uses {lab_name}, a name from the lab, instead of its arguments "
            "y_true and y_prob. Compute everything from the arguments.")

    # Diagnosis 2's probabilities, as Diagnosis 2 left them
    try:
        assert np.allclose(p_platt, platt_bal.predict_proba(X_eval)[:, 1])
        assert np.allclose(p_bal_eval, forest_bal.predict_proba(X_eval)[:, 1])
    except Exception:
        raise AssertionError(
            "p_platt or p_bal_eval is no longer what Diagnosis 2 computed: a cell gave that name to "
            "something else. Give your own results other names, then rerun the notebook from the "
            "top.") from None

    def run(where, y_true, y_prob, n_bins=None):
        """Call calibration_error on copies; return one number, or stop with a plain message."""
        args = (y_true.copy(), y_prob.copy()) + (() if n_bins is None else (n_bins,))
        try:
            got = calibration_error(*args)
        except TypeError as err:
            if "scalar" in str(err):
                raise AssertionError(
                    f"{where}, calibration_error stopped with TypeError: {err}. int(...) turns one "
                    "number into a whole number, not an array: use .astype(int), as Diagnosis 1 "
                    "did.") from None
            raise AssertionError(f"{where}, calibration_error stopped with TypeError: "
                                 f"{str(err).rstrip('.')}.") from None
        except IndexError as err:
            raise AssertionError(
                f"{where}, calibration_error stopped with IndexError: {err}. A probability of "
                "exactly 1.0 gets bin number n_bins, one past the top bin: put it in the top bin, "
                "as Diagnosis 1 did.") from None
        except Exception as err:
            raise AssertionError(f"{where}, calibration_error stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        assert got is not None, "calibration_error gives back nothing: use return, not print."
        try:
            value = float(got)
        except (TypeError, ValueError):
            raise AssertionError(f"{where}, calibration_error gives back {shown(got)}, not one "
                                 "number. Return the weighted average of the gaps.") from None
        assert not np.isnan(value), (
            f"{where}, calibration_error gives nan, which is what the mean of no rows gives. Leave "
            "out the bins that hold no predictions.")
        assert value > -1e-9, (
            f"{where}, calibration_error gives {value:.4f}, below zero. Take each gap without its "
            "sign.")
        return value

    # 1. Platt's probabilities from Diagnosis 2: three slips in the bins show up here first
    where = "On Platt's probabilities"
    platt = run(where, y_eval, p_platt)
    PLATT_SLIPS = {
        0.0267755482: "come from groups of equal size, as strategy=\"quantile\" makes. Use bins of "
                      "equal width on [0, 1]",
        0.0337912491: "come from cutting the range of the predictions, from the smallest to the "
                      "largest, into equal widths, as pd.cut(p, n_bins) or np.histogram without "
                      "range=(0, 1) do. Cut [0, 1] itself",
        0.0123092243: "round p * n_bins to the nearest whole number. np.floor rounds down, which is "
                      "what gives bins of equal width",
    }
    for slip, cause in PLATT_SLIPS.items():
        assert not near(platt, slip), f"{where}, calibration_error gives {platt:.4f}. Your numbers {cause}."

    # 2. the small example from the task
    toy_y = np.array([0, 0, 1, 1, 0, 1, 0])
    toy_p = np.array([0.05, 0.15, 0.15, 0.35, 0.45, 0.85, 1.0])
    where = "On the small example"
    toy = run(where, toy_y, toy_p)
    LAB = {"raw": 0.261257472915, "Platt": 0.0242251036746, "Platt, 5 bins": 0.0109929826221}
    for label, value in LAB.items():
        assert not near(toy, value), (
            f"{where}, calibration_error gives {toy:.4f}, the answer for the balanced forest's "
            f"{label} probabilities. Does it use its arguments, y_true and y_prob?")
    assert abs(toy) > 1e-9, (
        f"{where}, calibration_error gives 0, yet no bin's default rate equals its mean prediction. "
        "Does it keep each gap's sign, so that gaps above and below the diagonal cancel, or compare "
        "the overall means instead of each bin's? Take each bin's gap without its sign.")
    TOY_SLIPS = {
        0.441666666667: "average the bins' gaps without weighting them. Weight each gap by the share "
                        "of all the rows that falls in its bin",
        0.270714285714: "square the gaps. Take each gap without its sign, not squared",
        0.520302110042: "take the square root of the weighted mean of the squared gaps. Take each gap "
                        "without its sign, not squared",
        3.0: "weight each gap by the number of rows in its bin. Weight it by the share of all the rows",
        0.265: "add up the gaps and divide by the number of bins. Weight each gap by the share of all "
               "the rows that falls in its bin",
        2.65: "add up the gaps without weighting them. Weight each gap by the share of all the rows "
              "that falls in its bin",
        42.8571428571: "are a percentage. Return a share between 0 and 1",
        0.285714285714: "leave out the prediction of exactly 1.0, which gets bin number n_bins, one past "
                        "the top bin. Put it in the top bin, as Diagnosis 1 did",
        0.333333333333: "leave out the prediction of exactly 1.0 and weight by the rows that are left. "
                        "Put it in the top bin, as Diagnosis 1 did",
        0.471428571429: "come from groups of equal size, as strategy=\"quantile\" makes. Use bins of "
                        "equal width on [0, 1]",
        0.3: "round p * n_bins to the nearest whole number. np.floor rounds down, which is what gives "
             "bins of equal width",
        0.342142857143: "are the Brier score. Average the gaps between each bin's default rate and its "
                        "mean prediction",
    }
    for slip, cause in TOY_SLIPS.items():
        assert not near(toy, slip), f"{where}, calibration_error gives {toy:.4f}. Your numbers {cause}."
    assert near(toy, 3 / 7), (
        f"{where}, calibration_error gives {toy:.4f}; it should give 0.4286. Work the example by hand "
        "and compare each step with the task: the bin numbers, the gap in each bin, and the weights.")

    # 3. the balanced forest's raw probabilities, and Platt's, on the evaluation rows
    where = "On the balanced forest's raw probabilities"
    raw = run(where, y_eval, p_bal_eval)
    assert near(raw, LAB["raw"]), (
        f"{where}, calibration_error gives {raw:.4f}; it should give 0.2613. It passed the small "
        "example, so look for a step that only matters with many rows.")
    where = "On Platt's probabilities"
    assert near(platt, LAB["Platt"]), (
        f"{where}, calibration_error gives {platt:.4f}; it should give 0.0242. Compare each step with "
        "the task: the bin numbers, the gap in each bin, and the weights.")

    # 4. five bins
    where = "With n_bins=5"
    five = run(where, y_eval, p_platt, 5)
    assert not near(five, LAB["Platt"]), (
        f"{where}, calibration_error gives the answer for 10 bins. Does it use its n_bins argument "
        "everywhere, or the number 10?")
    toy_five = run("On the small example with n_bins=5", toy_y, toy_p, 5)
    where = "On the small example with n_bins=20"
    toy_twenty = run(where, toy_y, toy_p, 20)
    assert not near(toy_twenty, 1.85 / 7), (
        f"{where}, calibration_error gives {toy_twenty:.4f}: it leaves out every bin from 10 up. Does "
        "the loop run over a fixed count, range(10), instead of range(n_bins)?")
    assert near(toy_twenty, 3 / 7), (
        f"{where}, calibration_error gives {toy_twenty:.4f}; it should give 0.4286, as with 10 bins "
        "(work it by hand). Does every step use n_bins, with no fixed count such as 10?")
    assert near(five, LAB["Platt, 5 bins"]) and near(toy_five, 2.6 / 7), (
        f"{where}, calibration_error gives {five:.4f} on Platt's probabilities and {toy_five:.4f} on "
        "the small example; they should be 0.0110 and 0.3714. Does every step use n_bins?")

    print(f"Passed: calibration_error matches. Small example: {toy:.4f} (5 bins: {toy_five:.4f}).")
    print(f"Balanced forest, raw: {raw:.4f}; after Platt scaling: {platt:.4f} (5 bins: {five:.4f}).")

---

## Ship It: `src/classify_tools.py` (10 min)

Here is the module, finished except for one function. Its design rule is the one this lab
argued: discrimination and calibration are measured by separate functions, so that neither
number can stand in for the other. `discrimination` returns the AUC and the average precision;
`calibration` returns the Brier score, the mean predicted probability beside the observed default
rate, and your expected calibration error; `report` lays them out in a table, one row per set of
probabilities, with the cost of the rule threshold from Diagnosis 3.

**Your task.** Paste your `calibration_error` from Write it yourself, unchanged, at the end of the
first cell, where it says so, then run the two cells and compare the output with the expected
output below. The first saves the module as a file; the second imports it and reports Diagnosis 2's
three sets of probabilities on the evaluation rows.

**Reading the cells.** `%%writefile src/classify_tools.py` saves the cell as that file instead of
running it. The notes after a colon or `->` (`n_bins: int`, `-> dict`) only say what type each
argument and result is. `metrics.average_precision_score` is the area under the precision-recall
curve; unlike AUC, whose no-skill value is 0.5, its no-skill value is the prevalence. `{**a, **b}`
builds one dictionary from the entries of two. In `report`, `{name: ... for name, p in
probs.items()}` builds a dictionary with one entry per set of probabilities, `pd.DataFrame(rows)`
makes each entry a column, and `.T` turns the columns into rows. In the second cell, `import
classify_tools` loads the file as a module, and `hasattr(classify_tools, "calibration_error")` asks
whether the module defines that name.

In [ ]:
%%writefile src/classify_tools.py
"""Classification reports that keep discrimination and calibration apart (ECON 5200 Lab 17).

A model can rank well and still give probabilities that are miscalibrated, which matters to any rule
that compares a probability with a threshold. So the two are measured by separate functions.
"""
import numpy as np
import pandas as pd
from sklearn import metrics


def discrimination(y_true, y_prob) -> dict:
    """How well the probabilities rank: AUC and average precision. Order-keeping changes leave both."""
    return {"auc": metrics.roc_auc_score(y_true, y_prob),
            "avg_precision": metrics.average_precision_score(y_true, y_prob)}


def calibration(y_true, y_prob, n_bins: int = 10) -> dict:
    """Whether the probabilities are calibrated: Brier, mean predicted against observed, ECE."""
    return {"brier": metrics.brier_score_loss(y_true, y_prob),
            "mean_predicted": float(np.mean(y_prob)),
            "observed_rate": float(np.mean(y_true)),
            "ece": calibration_error(y_true, y_prob, n_bins)}


def cost_per_1000(y_true, y_prob, threshold: float, fn_cost: float = 8, fp_cost: float = 1) -> float:
    """Cost per 1,000 decisions of flagging every row with a probability of at least threshold."""
    flag = y_prob >= threshold
    missed = np.sum(~flag & (y_true == 1))
    false_alarms = np.sum(flag & (y_true == 0))
    return 1000 * (fn_cost * missed + fp_cost * false_alarms) / len(y_true)


def report(y_true, probs: dict, fn_cost: float = 8, fp_cost: float = 1) -> pd.DataFrame:
    """One row per set of probabilities: discrimination, calibration, and the cost at t = C_FP/(C_FP+C_FN)."""
    rule_t = fp_cost / (fp_cost + fn_cost)
    rows = {name: {**discrimination(y_true, p), **calibration(y_true, p),
                   "cost_at_rule": cost_per_1000(y_true, p, rule_t, fn_cost, fp_cost)}
            for name, p in probs.items()}
    return pd.DataFrame(rows).T


# Paste your calibration_error function from Write it yourself below this line, unchanged.

In [ ]:
# GIVEN — run as-is, once your calibration_error is pasted into the module
# Import the module and report Diagnosis 2's three sets of probabilities on the evaluation rows
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import classify_tools
importlib.reload(classify_tools)   # reload, so a re-run of the %%writefile cell takes effect

if not hasattr(classify_tools, "calibration_error"):
    print("Paste your calibration_error from Write it yourself into the module cell above first, "
          "then run that cell and this one again.")
else:
    probs = {"random forest": forest.predict_proba(X_eval)[:, 1],
             "balanced forest": p_bal_eval,
             "balanced + Platt": p_platt}
    print(classify_tools.report(y_eval, probs).round(4).to_string())

**Expected output:**
```
                     auc  avg_precision   brier  mean_predicted  observed_rate     ece  cost_at_rule
random forest     0.8201         0.4485  0.0926          0.1303         0.1325  0.0196      502.5000
balanced forest   0.8178         0.4320  0.1710          0.3938         0.1325  0.2613      791.6667
balanced + Platt  0.8178         0.4320  0.0930          0.1435         0.1325  0.0242      508.3333
```
The `cost_at_rule` column is Diagnosis 3's cost at t = 1/9, unrounded.

---

## Extension (Optional — 20 min): Calibration Drift

The logistic model from Diagnosis 1 was fitted on loans of which 13.3% defaulted. Suppose that in
deployment the default rate rises to about 25%, while the way the features drive default stays
the same: only the intercept of the true model moves.

**The task.** In the cell below, and in new cells under it:
1. Draw 4,000 new loans as Diagnosis 1 did, with a new generator, `np.random.default_rng(7)`, and
   the intercept raised from −2.6 to −1.6.
2. Score them with `logit`. Print its AUC beside Diagnosis 1's 0.8389, the mean of its predicted
   probabilities beside the new default rate, and its `calibration_error`.
3. Correct the intercept: add the same number s to the log-odds of every prediction,
   log(p / (1 − p)), which is `np.log(p / (1 - p))` (`np.log` is the natural logarithm, the inverse
   of `np.exp`), and turn the result back into probabilities with 1 / (1 + e^(−z)), as
   Diagnosis 1 did. Choose s from the grid 0.00, 0.01, ..., 2.00 as the shift whose mean predicted
   probability is closest to the new default rate, the way Diagnosis 3 chose a threshold. Measure
   the corrected probabilities again, and compare s with the true change in the intercept.

In [ ]:
# YOUR TASK — optional: calibration drift

### Interpretation Questions

1. Explain why a rise in the default rate damages calibration, why step 3 leaves AUC exactly where it was, and whether the change in AUC from 0.8389 is a sign that the model got worse.
2. Step 3 needed the new default rate. Say what you would monitor in production to catch this drift, and what the correction assumes about how the features drive default.

*Your answers here:*

1.

2.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated 8,000 loans with a known default model and fitted a
  logistic regression and a random forest. Their AUCs were
  [YOUR VALUE] and [YOUR VALUE], but the forest's predictions never
  went above [YOUR VALUE]
* Grew the forest again on a balanced sample: AUC [YOUR VALUE],
  but a mean predicted default rate of [YOUR VALUE] against a true
  rate of [YOUR VALUE]
* Repaired it with Platt scaling fitted on held-out rows: the Brier
  score went from [YOUR VALUE] to [YOUR VALUE], with AUC unchanged
* Priced the miscalibration with a missed default costing 8 times a
  false alarm: [YOUR VALUE] against [YOUR VALUE] per 1,000 decisions
* Wrote my own expected calibration error ([YOUR VALUE] raw,
  [YOUR VALUE] after Platt) and shipped a module that reports
  discrimination and calibration separately

**Please write a README.md entry including:**
1. Project Title: Classification: Ranking Is Not Calibration
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person, with no buzzwords. Use only
the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab17-classification`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab17-classification`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab17-classification`, branch `main`,
   commit message `Lab 17 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/classify_tools.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.